# NSDDD v3 Getting Started Tutorial

Welcome! This notebook demonstrates how to use the National Security Documents Dataset (NSDDD) v3 for semantic analysis.

## What This Tutorial Covers

1. **Loading the data model** – Access embeddings and segment data
2. **Running semantic searches** – Find documents matching your queries
3. **Filtering results** – By country, year, document type
4. **Analysing results** – Clustering and exploring patterns
5. **Advanced techniques** – Batch searches, custom analyses

## Dataset Overview

- **660 documents** from **119 countries**
- **1987–2025** (38 years)
- **726,307 segments** (paragraph-level text units)
- **768-dimensional embeddings** using sentence-transformers/all-mpnet-base-v2
- **All searches run locally** – no internet or API keys needed

## Getting Help

- See `documentation/SEMANTIC_SEARCH_GUIDE.md` for complete methodology
- See `documentation/NSDDD_v3_LAUNCH_REPORT.md` for dataset composition
- Each section includes explanations and examples

---

Let's begin! 👇


## 1. Loading the Data Model

First, we'll load the pre-computed embeddings and segment data.

**Note**: The embedding file is 11 GB. Loading takes 90–120 seconds.


In [ ]:
import json
import numpy as np
import time
from pathlib import Path
from collections import defaultdict

print('Loading NSDDD v3 data model...\n')

# Configuration
MODEL_DIR = Path('model')

# Verify files exist
required_files = [
    'segment_encodings.json',  # 11 GB embeddings
    'segments_dict.json',      # Segment text
    'encoded_segments.json',   # Segment IDs
    'documents_dict.json',     # Document metadata
    'countries_dict.json'      # Country metadata
]

print('Checking files...')
for filename in required_files:
    filepath = MODEL_DIR / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024**2)
        print(f'  ✓ {filename}: {size_mb:.1f} MB')
    else:
        print(f'  ✗ {filename}: NOT FOUND')
        raise FileNotFoundError(f'{filename} not found in model/ directory')

print('\n' + '='*60)
print('Loading embeddings (this takes 90–120 seconds)...')
print('='*60 + '\n')

# Load embeddings
start = time.time()
print('segment_encodings.json (11 GB)...', end=' ', flush=True)
with open(MODEL_DIR / 'segment_encodings.json', 'r') as f:
    segment_encodings = np.array(json.load(f))
elapsed = time.time() - start
print(f'✓ {elapsed:.1f}s')
print(f'  Shape: {segment_encodings.shape}')
print(f'  Type: {segment_encodings.dtype}\n')

# Load segments dictionary
print('segments_dict.json...', end=' ', flush=True)
with open(MODEL_DIR / 'segments_dict.json', 'r') as f:
    segments_dict = json.load(f)
print(f'✓')
print(f'  Segments: {len(segments_dict):,}\n')

# Load encoded segments list
print('encoded_segments.json...', end=' ', flush=True)
with open(MODEL_DIR / 'encoded_segments.json', 'r') as f:
    encoded_segments = json.load(f)
print(f'✓')
print(f'  Encoded segments: {len(encoded_segments):,}\n')

# Load documents
print('documents_dict.json...', end=' ', flush=True)
with open(MODEL_DIR / 'documents_dict.json', 'r') as f:
    documents_dict = json.load(f)
print(f'✓')
print(f'  Documents: {len(documents_dict):,}\n')

# Load countries
print('countries_dict.json...', end=' ', flush=True)
with open(MODEL_DIR / 'countries_dict.json', 'r') as f:
    countries_dict = json.load(f)
print(f'✓')
print(f'  Countries: {len(countries_dict):,}\n')

print('='*60)
print('✓ All files loaded successfully!')
print('='*60)


## 2. Running Semantic Searches

Now we'll encode a query and search for similar segments.


In [ ]:
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

# Load encoder
print('Loading sentence-transformers model...')
print('(Only needed once per session)\n')

encoder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

print('✓ Model loaded\n')

# Define search query
query = 'Cyber threats to critical infrastructure'

print(f'Query: "{query}"\n')

# Encode query
print('Encoding query...')
query_embedding = encoder.encode([query])[0]
print(f'✓ Query encoded to {len(query_embedding)} dimensions\n')

# Compute similarities
print('Computing similarity scores for all segments...')
print('(This may take 30–60 seconds)\n')

similarities = []
for i, encoding in enumerate(segment_encodings):
    sim = 1 - cosine(query_embedding, encoding)
    similarities.append(sim)
    if (i + 1) % 100000 == 0:
        print(f'  Processed {i + 1:,} / {len(segment_encodings):,}')

similarities = np.array(similarities)

print('\n✓ Similarity computation complete')
print(f'  Min similarity: {similarities.min():.3f}')
print(f'  Max similarity: {similarities.max():.3f}')
print(f'  Mean similarity: {similarities.mean():.3f}')


In [ ]:
# Get top results
threshold = 0.7
top_count = 20

# Filter by threshold
matching_indices = np.where(similarities >= threshold)[0]
print(f'Results with similarity ≥ {threshold}: {len(matching_indices):,}\n')

# Get top N
top_indices = np.argsort(similarities)[::-1][:top_count]

print(f'Top {top_count} Results:')
print('='*80)
print(f'{"Rank":<5} {"Segment ID":<15} {"Similarity":<12} {"Country":<20} {"Text Preview"}')
print('─'*80)

for rank, idx in enumerate(top_indices, 1):
    segment_id = encoded_segments[idx]
    similarity = similarities[idx]

    # Parse segment ID
    doc_id, seg_num = segment_id.split('/')

    # Get country
    if doc_id in documents_dict:
        doc = documents_dict[doc_id]
        country = doc.get('country', 'Unknown')
        year = doc.get('year', '????')
    else:
        country = 'Unknown'
        year = '????'

    # Get text preview
    if segment_id in segments_dict:
        text = segments_dict[segment_id][:40].replace('\n', ' ')
    else:
        text = '[Not found]'

    print(f'{rank:<5} {segment_id:<15} {similarity:<12.3f} {country:<20} {text}...')

print('='*80)


## 3. Filtering Results by Country, Year, Document Type

We can filter search results using metadata (country, publication year, document type).


In [ ]:
# Explore metadata structure
print('Sample document metadata:\n')

# Get a few examples
for doc_id in list(documents_dict.keys())[:3]:
    doc = documents_dict[doc_id]
    print(f'Document ID {doc_id}:')
    for key, value in sorted(doc.items()):
        print(f'  {key}: {value}')
    print()

# Summary statistics
print('\nMetadata Summary:')
print('─'*60)

years = set()
countries = set()
doc_types = set()

for doc in documents_dict.values():
    if 'year' in doc:
        years.add(doc['year'])
    if 'country' in doc:
        countries.add(doc['country'])
    if 'type' in doc:
        doc_types.add(doc['type'])

print(f'Countries: {len(countries):,}')
print(f'Years: {min(years) if years else "N/A"} – {max(years) if years else "N/A"}')
print(f'Document types: {len(doc_types)}')
print(f'  Types: {", ".join(sorted(doc_types))}')


In [ ]:
# Filter results by country
print('Filtering results by country\n')

country_filter = 'United States'  # Change this to filter by different country

print(f'Filter: Country = {country_filter}\n')

# Build index of documents by country
docs_by_country = defaultdict(list)
for doc_id, doc in documents_dict.items():
    if 'country' in doc:
        docs_by_country[doc['country']].append(doc_id)

print(f'Documents from {country_filter}: {len(docs_by_country[country_filter])}\n')

# Filter results
filtered_indices = []
for idx in top_indices:
    segment_id = encoded_segments[idx]
    doc_id, _ = segment_id.split('/')

    if doc_id in documents_dict:
        if documents_dict[doc_id].get('country') == country_filter:
            filtered_indices.append(idx)

print(f'Results from {country_filter}:')
print('='*80)
print(f'{"Rank":<5} {"Segment ID":<15} {"Similarity":<12} {"Year":<6} {"Text Preview"}')
print('─'*80)

for rank, idx in enumerate(filtered_indices[:10], 1):
    segment_id = encoded_segments[idx]
    similarity = similarities[idx]
    doc_id, _ = segment_id.split('/')

    doc = documents_dict[doc_id]
    year = doc.get('year', '????')

    if segment_id in segments_dict:
        text = segments_dict[segment_id][:40].replace('\n', ' ')
    else:
        text = '[Not found]'

    print(f'{rank:<5} {segment_id:<15} {similarity:<12.3f} {year:<6} {text}...')

print('='*80)


## 4. Extracting Full Segment Text and Metadata

Extract complete text and all metadata for top results.


In [ ]:
# Extract full results with complete information
print('Top 5 results with full details:\n')
print('='*80 + '\n')

for rank, idx in enumerate(top_indices[:5], 1):
    segment_id = encoded_segments[idx]
    similarity = similarities[idx]

    doc_id, seg_num = segment_id.split('/')
    doc = documents_dict[doc_id]

    print(f'Result {rank}')
    print(f'  Segment ID: {segment_id}')
    print(f'  Similarity: {similarity:.4f}')
    print(f'  Document: {doc.get("title", "Unknown")}')
    print(f'  Country: {doc.get("country", "Unknown")}')
    print(f'  Year: {doc.get("year", "????")}')
    print(f'  Type: {doc.get("type", "Unknown")}')

    # Get full segment text
    if segment_id in segments_dict:
        text = segments_dict[segment_id]
        print(f'\n  Full Text:')
        print(f'  {"-"*76}')
        # Word wrap text
        words = text.split()
        line = '  '
        for word in words:
            if len(line) + len(word) + 1 > 80:
                print(line)
                line = '  '
            line += word + ' '
        if line.strip() != '':
            print(line)
        print(f'  {"-"*76}')
    else:
        print('  Text: [Not found]')

    print()


## 5. Running Multiple Queries

Search for multiple topics and compare results.


In [ ]:
# Define multiple search queries
queries = [
    'Cyber threats to critical infrastructure',
    'Climate change as a national security threat',
    'Supply chain security',
    'Terrorism and non-state actors'
]

print('Running multiple semantic searches...\n')

results = {}

for query in queries:
    # Encode query
    query_embedding = encoder.encode([query])[0]

    # Compute similarities
    sims = []
    for encoding in segment_encodings:
        sim = 1 - cosine(query_embedding, encoding)
        sims.append(sim)

    # Get top 10
    top_indices = np.argsort(sims)[::-1][:10]
    top_sims = [sims[i] for i in top_indices]

    results[query] = {
        'indices': top_indices,
        'similarities': top_sims,
        'count_above_threshold': sum(1 for s in sims if s >= 0.7)
    }

# Display results
print('='*80)
print(f'{"Query":<40} {"High Confidence (≥0.7)":<25} {"Top Similarity"}')
print('='*80)

for query, result in results.items():
    count = result['count_above_threshold']
    top_sim = result['similarities'][0] if result['similarities'] else 0
    query_short = query[:40]
    print(f'{query_short:<40} {count:<25} {top_sim:.3f}')

print('='*80)


## 6. Country-Level Analysis

Explore which countries have documents and basic statistics.


In [ ]:
# Build country statistics
country_stats = defaultdict(lambda: {'count': 0, 'years': set()})

for doc_id, doc in documents_dict.items():
    country = doc.get('country', 'Unknown')
    year = doc.get('year')

    country_stats[country]['count'] += 1
    if year:
        country_stats[country]['years'].add(year)

# Sort by document count
sorted_countries = sorted(country_stats.items(), key=lambda x: x[1]['count'], reverse=True)

print(f'Document Coverage by Country\n')
print(f'{"Country":<30} {"Documents":<12} {"Year Range"}')
print('='*80)

for country, stats in sorted_countries[:20]:
    doc_count = stats['count']
    years = stats['years']
    if years:
        year_range = f'{min(years)}–{max(years)}'
    else:
        year_range = 'N/A'

    print(f'{country:<30} {doc_count:<12} {year_range}')

print('\n...')
print(f'\nTotal countries with documents: {len(country_stats)}')


## 7. Advanced: Comparing Threat Perceptions Across Countries

Compare how different countries frame security threats.


In [ ]:
# Compare how two countries discuss a specific threat
threat_query = 'Cyber threats to critical infrastructure'

countries_to_compare = ['United States', 'China', 'Russia']

print(f'Comparing threat perceptions: "{threat_query}"\n')
print(f'Countries: {", ".join(countries_to_compare)}\n')

# Encode the threat query
threat_embedding = encoder.encode([threat_query])[0]

# For each country, find the highest similarity segment
country_results = {}

for country in countries_to_compare:
    docs_ids = docs_by_country[country]

    country_max_sim = -1
    country_best_segment = None
    country_best_doc = None

    for doc_id in docs_ids:
        # Find segments from this document
        for i, seg_id in enumerate(encoded_segments):
            if seg_id.split('/')[0] == doc_id:
                sim = 1 - cosine(threat_embedding, segment_encodings[i])
                if sim > country_max_sim:
                    country_max_sim = sim
                    country_best_segment = seg_id
                    country_best_doc = doc_id

    if country_best_segment:
        country_results[country] = {
            'similarity': country_max_sim,
            'segment_id': country_best_segment,
            'doc_id': country_best_doc
        }

# Display comparison
print('='*80)
print(f'{"Country":<20} {"Max Similarity":<20} {"Document"}')
print('='*80)

for country in countries_to_compare:
    if country in country_results:
        result = country_results[country]
        doc = documents_dict[result['doc_id']]
        print(f'{country:<20} {result["similarity"]:<20.3f} {doc.get("title", "Unknown")[:40]}')
    else:
        print(f'{country:<20} No documents found')

print('='*80)


## Next Steps

### Continue Learning

1. **Explore different queries** – Try searching for topics relevant to your research
2. **Filter by country and year** – Compare how threats are framed across regions and time
3. **Read the documentation** – See `documentation/SEMANTIC_SEARCH_GUIDE.md` for advanced techniques
4. **Cluster results** – Group similar segments for pattern analysis

### Customisation Ideas

- Modify search thresholds to get more/fewer results
- Batch process multiple queries
- Analyse temporal patterns (how threat discourse changes over time)
- Compare threat frames across regions

### Important Notes

- All searches run **locally** on your computer – no internet or API keys needed
- The 11GB embedding file is accessed directly from disk
- You can modify the model files directory path if needed
- Results are reproducible – same query always produces same results

### Citation

If you publish research using NSDDD v3, cite:

```
Gardner, A. N. (2025). National Security Documents Dataset (NSDDD) Version 3.
University of Edinburgh. https://doi.org/10.7488/ds/[DOI_TO_BE_ADDED]
```

---

Happy researching! For questions or issues, see the included documentation. 🔍
